In [43]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

## 0. Setup & chargement

In [44]:
df = pd.read_csv(r"..\data\processed\final.csv")
df = df.drop(columns='Unnamed: 0')
df['date'] = pd.to_datetime(df['date'])

In [45]:
df.dtypes

client_id             object
sex                   object
birth                  int64
id_prod               object
date          datetime64[ns]
session_id            object
price                float64
categ                  int64
dtype: object

In [46]:
# identification des clients BtoB
ca_par_clients = df.groupby('client_id')['price'].sum().sort_values(ascending=False).reset_index()
ca_par_clients['segment'] = np.where(ca_par_clients['price'] > 50000, 'BtoB', 'BtoC')
df['segment'] = df['client_id'].map(ca_par_clients.set_index('client_id')['segment'])
df.shape

(687534, 9)

In [47]:
df_btoc = df.query("segment == 'BtoC'")
df_btoc.shape

(640734, 9)

In [48]:
df_btoc.head()

,client_id,sex,birth,id_prod,date,session_id,price,categ,segment
0,c_4410,f,1967,1_483,2021-03-13 21:35:55.949042,s_5913,15.99,1,BtoC
1,c_4410,f,1967,0_1111,2021-03-22 01:27:49.480137,s_9707,19.99,0,BtoC
2,c_4410,f,1967,1_385,2021-03-22 01:40:22.782925,s_9707,25.99,1,BtoC
3,c_4410,f,1967,0_1455,2021-03-22 14:29:25.189266,s_9942,8.99,0,BtoC
4,c_4410,f,1967,0_1420,2021-03-22 22:31:25.825764,s_10092,11.53,0,BtoC


## 1. CA dans le temps

### 1.1 CA mensuel + moyenne mobile

In [49]:
# Par mois

df_ca = df_btoc.set_index('date')
df_ca = df_ca.groupby(pd.Grouper(freq='ME'))['price'].sum().to_frame()
df_ca = df_ca.rename(columns={'price' : 'ca'})
df_ca['moy_mobile'] = round(df_ca['ca'].rolling(window=3).mean(), 2)
df_ca.head()

# ------------------------

# Par jours

df_ca_p = df_btoc.set_index('date')
df_ca_p = df_ca_p.groupby(pd.Grouper(freq='D'))['price'].sum().to_frame()
df_ca_p = df_ca_p.rename(columns={'price' : 'ca'})
df_ca_p['moy_mobile_3j'] = round(df_ca_p['ca'].rolling(window=3).mean(), 2)
df_ca_p['moy_mobile_7j'] = round(df_ca_p['ca'].rolling(window=7).mean(), 2)
df_ca_p['moy_mobile_15j'] = round(df_ca_p['ca'].rolling(window=15).mean(), 2)
df_ca_p['moy_mobile_30j'] = round(df_ca_p['ca'].rolling(window=30).mean(), 2)

In [57]:
fig_ca = go.Figure()

fig_ca.add_trace(go.Scatter(
    x=df_ca.index,
    y=df_ca['ca'],
    name="CA",
    mode='lines+markers',
    marker=dict(size=5),
    line=dict(color="#58508d"),
    hovertemplate="%{y:,.0f} €"
))

fig_ca.add_trace(go.Scatter(
    x=df_ca.index,
    y=df_ca['moy_mobile'],
    name="Moyenne",
    mode='lines+markers',
    marker=dict(size=5),
    line=dict(color='#ff6361'),
    hovertemplate="%{y:,.0f} €"
))

# Habillage
fig_ca.update_layout(
    yaxis=dict(title="Chiffre d'affaire", showgrid=False),
    xaxis=dict(tickformat="%b %Y",),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    height=500,
    hovermode='x unified',
    hoverlabel=dict(font_size=12),
    separators=". ",
    title="Evolution du chiffre d'affaire"
)

fig_ca.show()

In [58]:
fig_ca_p = go.Figure()

fig_ca_p.add_trace(go.Scatter(
    x=df_ca_p.index,
    y=df_ca_p['ca'],
    name="CA Journalier",
    mode='lines',
    line=dict(color="#D3D3D3"),
    hovertemplate="%{y:,.0f} €"
))

fig_ca_p.add_trace(go.Scatter(
    x=df_ca_p.index,
    y=df_ca_p['moy_mobile_7j'],
    name="Moyenne 7j",
    mode='lines',
    line=dict(color='#7a5195'),
    hovertemplate="%{y:,.0f} €"
))

fig_ca_p.add_trace(go.Scatter(
    x=df_ca_p.index,
    y=df_ca_p['moy_mobile_30j'],
    name="Moyenne 30j",
    mode='lines',
    line=dict(color='#ffa600'),
    hovertemplate="%{y:,.0f} €"
))

# Habillage
fig_ca_p.update_layout(
    yaxis=dict(title="Chiffre d'affaire", showgrid=False),
    xaxis=dict(tickformat="%d %b %Y"),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    height=600,
    hovermode='x unified',
    hoverlabel=dict(font_size=12),
    separators=". ",
    title="Evolution du chiffre d'affaire"
)

fig_ca_p.show()

### 1.2 CA par catégorie

In [59]:
# periode

df_categ_p = df_btoc.set_index('date')
df_categ_p = df_categ_p.groupby([pd.Grouper(freq='ME'), 'categ'])['price'].sum().to_frame().reset_index()
df_categ_p = df_categ_p.rename(columns={'price' : 'ca'})
df_categ_p = df_categ_p.pivot(index='date', columns='categ', values='ca')
df_categ_p.head()

# -------------------------

# cumul

df_categ = df_btoc.set_index('date')
df_categ = df_categ.groupby('categ')['price'].sum().reset_index()
df_categ = df_categ.rename(columns={'price' : 'ca'})
df_categ.head()


,categ,ca
0,0,4119200.69
1,1,4520101.86
2,2,2504064.46


In [60]:
fig_categ = go.Figure()

fig_categ.add_trace(go.Bar(
    name='0',
    x=df_categ_p.index,
    y=df_categ_p[0],
    hovertemplate="%{y:,.0f} €",
    marker_color='#003f5c'
))
fig_categ.add_trace(go.Bar(
    name='1',
    x=df_categ_p.index,
    y=df_categ_p[1],
    hovertemplate="%{y:,.0f} €",
    marker_color='#ffa600'
))
fig_categ.add_trace(go.Bar(
    name='2',
    x=df_categ_p.index,
    y=df_categ_p[2],
    hovertemplate="%{y:,.0f} €",
    marker_color='#bc5090'
))


# Habillage
fig_categ.update_layout(
    yaxis=dict(title="Chiffre d'affaire", showgrid=False),
    xaxis=dict(tickformat="%b %Y",),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    height=500,
    hovermode='x unified',
    hoverlabel=dict(font_size=12),
    separators=". ",
    title="Evolution du chiffre d'affaire par catégories",
    # barmode='stack'
)

fig_categ.show()

In [61]:
fig_categ = go.Figure()

fig_categ.add_trace(go.Bar(
    name='CA par catégorie',
    x=df_categ.index.astype(str),
    y=df_categ['ca'],
    hovertemplate="%{y:,.0f} €",
    marker_color=['#003f5c', '#ffa600', '#bc5090']
))

# Habillage
fig_categ.update_layout(
    yaxis=dict(title="Chiffre d'affaire", showgrid=False),
    legend=dict(orientation='h', y=1.1, x=0.5, xanchor='center'),
    height=500,
    hovermode='x unified',
    hoverlabel=dict(font_size=12),
    separators=". ",
    title="Evolution du chiffre d'affaire par catégories"
)

fig_categ.show()

### 1.3 Métriques clés / mois (clients, transactions, produits)

## 2. Zoom catalogue

### 2.1 Top / Flop produits

### 2.2 Répartition CA par catégorie

## 3. Profils clients

### 3.1 Répartition BtoB vs BtoC

In [63]:
df_donuts = df.groupby('segment')['price'].sum().sort_values(ascending=False).reset_index()
df_donuts

,segment,price
0,BtoC,11143367.01
1,BtoB,884296.09


In [108]:
donuts_btob = go.Figure()

donuts_btob.add_trace(go.Pie(
    labels=df_donuts['segment'],
    values=df_donuts['price'],
    hole=0.65,
    hovertemplate="%{value:,.0f} €<extra></extra>",
    marker=dict(colors=['#58508d', '#ff6361']),
    textinfo='label+percent',
    title=dict(text="Repartition du CA<br>entre BtoB et BtoC", font=dict(size=18))
))

donuts_btob.update_layout(
    separators=". ",
    width=440,
    margin=dict(t=15, b=15, l=15, r=15),
    showlegend=False
)

### 3.2 Courbe de Lorenz